# 03 - Concept Injection

Steer generation with an extracted concept vector and inspect the effect on the
model's answers.

## What steering does

Phase 2 gave us a unit direction in the residual stream for each cultural
concept. Injection is the mirror image of extraction: instead of *reading* the
activation at `blocks.{layer}.hook_resid_post`, we *write* to it, adding a scaled
copy of the concept vector as the activation flows past:

$$
h^{(\ell)} \leftarrow h^{(\ell)} + \alpha \cdot v_{\text{concept}}
$$

Because $v$ is normalised to unit length, the coefficient $\alpha$ is measured
in residual stream norms and is the only magnitude knob:

| $\alpha$ | Effect |
| --- | --- |
| $\alpha > 0$ | Amplify the concept |
| $\alpha = 0$ | Exactly the unsteered model |
| $\alpha < 0$ | Suppress the concept |

The offset is added at **every sequence position**, so the whole context is
nudged rather than only the final token. Shapes work out by ordinary
broadcasting: the vector is `(d_model,)` and the activation is `(batch, seq,
d_model)`.

## Why the context manager matters

A forward hook mutates the model until it is removed. Forget to remove one and
every later call in the session is silently steered — the kind of bug that
produces a whole afternoon of results that quietly mean nothing. So the
recommended entry point is `engine.steering(...)`, a context manager that
removes exactly the hooks it added, on the way out, even if the body raises:

```python
with engine.steering("diyafa_001", strength=2.0):
    output = engine.model.generate(prompt)   # steered
# hooks are gone here, whatever happened above
```

`inject_vector()` / `remove_hooks()` are still available for the rare case where
steering has to outlive a block of code.

## Setup

This notebook runs end to end on CPU against a tiny randomly-initialised model,
so it executes anywhere. Set `USE_TINY_MODEL = False` to steer the real base
model (a GPU is strongly recommended).

With random weights the *text* below is meaningless — what is verifiable is the
mechanism: the activation shifts by exactly the amount we asked for, and the
hooks are gone afterwards.

In [ ]:
import os
from pathlib import Path

import torch
from dotenv import load_dotenv

from src.models.rep_engine import RESID_POST_HOOK, CulturalRepE
from src.utils.evaluation import evaluate_steering, generate_steered, summarize_sweep

load_dotenv()

USE_TINY_MODEL = True  # set to False to use BASE_MODEL_NAME from .env

DATASET_PATH = Path("../data/datasets/cultural_concepts.jsonl")
CONCEPT = "diyafa_001"
PROMPT = "What should I do when a guest arrives unannounced?"

In [ ]:
if USE_TINY_MODEL:
    from transformer_lens import HookedTransformer, HookedTransformerConfig

    tiny_cfg = HookedTransformerConfig(
        n_layers=4,
        d_model=16,
        n_ctx=64,
        d_head=4,
        n_heads=4,
        d_mlp=32,
        act_fn="gelu",
        d_vocab=50257,
        tokenizer_name="sshleifer/tiny-gpt2",
        device="cpu",
    )
    engine = CulturalRepE(
        model_name="tiny-gpt2-random",
        device="cpu",
        dtype="float32",
        dataset_path=DATASET_PATH,
    )
    engine.model = HookedTransformer(tiny_cfg)
    engine.tokenizer = engine.model.tokenizer
else:
    engine = CulturalRepE(
        model_name=os.environ.get("BASE_MODEL_NAME", "meta-llama/Meta-Llama-3-8B-Instruct"),
        device=os.environ.get("DEVICE", "cuda"),
        dtype=os.environ.get("DTYPE", "bfloat16"),
        hf_token=os.environ.get("HF_TOKEN") or None,  # from .env, never inline
        dataset_path=DATASET_PATH,
    )
    engine.load_model()

vector = engine.extract_vector(CONCEPT)
layer = engine.extraction_layers[CONCEPT]
print(f"concept {CONCEPT}: vector {tuple(vector.shape)} extracted at layer {layer}")

## The hook lifecycle, made visible

`active_hook_names` reports what is currently attached. Watch it go from empty,
to one hook inside the block, back to empty on the way out.

In [ ]:
print("before :", engine.active_hook_names)

with engine.steering(CONCEPT, strength=2.0) as handles:
    print("inside :", engine.active_hook_names)
    print("        ", handles[0])

print("after  :", engine.active_hook_names)

### Cleanup survives an exception

This is the property that makes the context manager worth using over a manual
inject/remove pair.

In [ ]:
try:
    with engine.steering(CONCEPT, strength=2.0):
        raise RuntimeError("something went wrong mid-generation")
except RuntimeError as exc:
    print("caught:", exc)

print("hooks after the exception:", engine.active_hook_names)

## Verify the injection numerically

Read the residual stream with and without steering. The difference must be
exactly `strength * vector` at every position — this is the assertion that
proves the mechanism, independent of whether the generated text looks
plausible.

In [ ]:
def resid_post(prompt: str, layer: int) -> torch.Tensor:
    """Read the residual stream after `layer` for a single prompt."""
    hook_name = RESID_POST_HOOK.format(layer=layer)
    with torch.no_grad():
        _, cache = engine.model.run_with_cache(
            engine.model.to_tokens([prompt]), names_filter=hook_name, return_type=None
        )
    return cache[hook_name].clone()


STRENGTH = 3.0
baseline = resid_post(PROMPT, layer)

with engine.steering(CONCEPT, strength=STRENGTH):
    steered = resid_post(PROMPT, layer)

delta = steered - baseline
expected = (STRENGTH * vector).expand_as(delta)

print("shift matches strength * vector:", torch.allclose(delta, expected, atol=1e-5))
print("same offset at every position  :", torch.allclose(delta, delta[0, 0].expand_as(delta), atol=1e-5))
print("model restored after the block :", torch.equal(resid_post(PROMPT, layer), baseline))

## A strength sweep

`evaluate_steering` walks a grid of strengths. At each point it enters the
steering context, generates a continuation, and measures the model's
cross-entropy **on the prompts themselves**.

That second number is the guardrail. It is computed on text the steering did not
produce, so a sharp rise means the injection is damaging the model's language
modelling rather than merely changing its topic. Steering that wins on cultural
grounding while wrecking fluency has not actually won.

In [ ]:
results = evaluate_steering(
    engine,
    CONCEPT,
    prompts=[PROMPT, "أكرم ضيافته"],
    strengths=[-2.0, -1.0, 0.0, 1.0, 2.0],
    max_new_tokens=12,
)

for strength, result in results.items():
    print(f"{strength:+.1f}  loss={result.mean_loss:8.4f}  perplexity={result.perplexity:12.2f}")

### Where does fluency break down?

Plot cross-entropy against strength. On a trained model this curve is typically
flat near zero and turns sharply upward past some threshold — that knee is the
usable steering range, and it is what you report in the paper rather than a
single hand-picked strength.

In [ ]:
from src.utils.visualization import plot_steering_sweep

summary = summarize_sweep(results)
plot_steering_sweep(
    summary["strengths"],
    summary["mean_losses"],
    metric_name="Mean cross-entropy (nats/token)",
)

### Qualitative side-by-side

Read the generations next to each other. With random weights this is noise; on a
real model this is where cultural steering either shows up as a change in *how*
the answer is framed, or reveals itself as degradation.

In [ ]:
for strength, result in results.items():
    text = result.generations[PROMPT]
    print(f"--- strength {strength:+.1f} " + "-" * 40)
    print(text)
    print()

## Injecting into several layers

A single layer is the conservative default. Spreading the same offset across
several mid-stack layers usually produces a stronger effect per unit of
strength, at the cost of a larger fluency hit — worth sweeping as its own
axis.

In [ ]:
mid = engine.n_layers // 2
layer_sets = [[mid], [mid - 1, mid], [mid - 1, mid, mid + 1]]

for layers in layer_sets:
    if max(layers) >= engine.n_layers or min(layers) < 0:
        continue
    with engine.steering(CONCEPT, strength=1.0, layers=layers):
        text = generate_steered(engine, PROMPT, max_new_tokens=10)
    print(f"layers {layers}: {text[len(PROMPT):].strip()!r}")

print("hooks left attached:", engine.active_hook_names)

## Next steps

- Sweep the layer set as well as the strength; report the knee of the fluency
  curve rather than one hand-picked coefficient.
- Have native speakers rate the generations blind for cultural grounding — the
  automatic loss metric here only detects damage, not success.
- Check that suppression ($\alpha < 0$) actually removes the concept rather
  than producing generic text.
- Compare against a prompt-engineering baseline: steering has to beat "answer
  with Arab hospitality in mind" to be worth the complexity.